### GERADOR DE DATASET (RF01)


In [ ]:
##importando bibliotecas

import csv
import json
import re
import os
import random
from datetime import datetime, timedelta
from collections import defaultdict

In [ ]:
# função para gerar um dataset

def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
    random.seed(seed)
    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)
    colunas = ["id_venda", "data_venda", "cliente", "produto",
               "categoria", "regiao", "quantidade", "preco_unitario"]

    with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()

        for i in range(n_registros):
            produto = random.choice(produtos)
            categoria = categorias[produto]
            quantidade = random.randint(1, 10)
            preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
            data = data_inicio + timedelta(days=random.randint(0, 364))
            data_txt = data.strftime("%Y-%m-%d")
            cliente = f"Cliente_{random.randint(1, 50):03d}"

            # --- sujeira proposital para a etapa de limpeza ---
            if random.random() < 0.05:
                quantidade = ""
            if random.random() < 0.04:
                preco = ""
            if random.random() < 0.06:
                produto = " " + produto + " "
            if random.random() < 0.03:
                data_txt = "DATA INVALIDA"
            if random.random() < 0.10:
                cliente = random.choice([
                    cliente.upper().replace("_", "-"),
                    cliente + "!!",
                    " " + cliente,
                    cliente.replace("Cliente_", "cliente#"),
                ])

            escritor.writerow({
                "id_venda": i + 1,
                "data_venda": data_txt,
                "cliente": cliente,
                "produto": produto,
                "categoria": categoria,
                "regiao": random.choice(regioes),
                "quantidade": quantidade,
                "preco_unitario": preco,
            })

    print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")

In [ ]:
#se ja existir um dataset chamado vendas.csv essa verificação pula etapa geração de csv

if not os.path.exists("vendas.csv"):
    gerar_dataset_vendas("vendas.csv")
else:
    print("vendas.csv já existe, geração pulada.")

### CARREGANDO O DATASET (RF01)

In [ ]:
def carregar_dataset(caminho_csv):
    """
    Carrega um dataset de vendas a partir de um arquivo CSV.
    Parâmetros:
        caminho_csv (str): caminho para o arquivo CSV.
    Retorna:
        list: lista de dicionários, um por registro.
    """
    try:
        with open(caminho_csv, "r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            registros = list(leitor)
        print(f"Arquivo '{caminho_csv}' carregado com sucesso.")
        return registros
    except FileNotFoundError:
        print(f"Erro: o arquivo '{caminho_csv}' não foi encontrado.")
        return []

registros = carregar_dataset("vendas.csv")

print()
print(f"Total de registros carregados: {len(registros)}")
print()
print(registros[0])

### INSPECIONANDO E DESCREVENDO OS DADOS (RF02)

In [ ]:
def inspecionar_dados(registros):
    """
    Exibe as informações estruturais do dataset carregado:
    total de registros, colunas, valores ausentes por coluna
    e os primeiros registros.
    Parâmetros:
        registros (list): lista de dicionários representando o dataset.
    Retorna:
        list: os mesmos registros recebidos (sem alteração).
    """
    if not registros:
        print("O dataset está vazio.")
        return registros

    total = len(registros)
    colunas = list(registros[0].keys())

    nulos = {coluna: 0 for coluna in colunas}
    for linha in registros:
        for coluna in colunas:
            if linha[coluna] == "":
                nulos[coluna] += 1

    print("=== INSPEÇÃO INICIAL DO DATASET ===")
    print()
    print(f"Total de registros: {total}")
    print(f"Total de colunas: {len(colunas)}")
    print(f"\nColunas: {colunas}")
    print(f"\nValores ausentes por coluna:\n{nulos}")
    print("\nPrimeiros 5 registros:")
    for linha in registros[:5]:
        print(linha)

    return registros

registros = inspecionar_dados(registros)

### LIMPANDO E TRATANDO OS DADOS (RF03)

In [ ]:
def limpar_dados(dataset):
    """ Limpa os dados do dataset, removendo espaços em branco, convertendo tipos e tratando valores ausentes.
    Parâmetros:
        dataset (list): lista de dicionários com os registros.
    Retorna:
        tupla: (lista de dicionários limpos, relatório de limpeza) """

    relatorio = {
        'iniciais': len(dataset),
        'removidos_data': 0,
        'removidos_nulos': 0,
        'clientes_padronizados': 0,
        'finais': 0
    }

    padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)
    registros_limpos = []

    for linha in dataset:
        linha = dict(linha)  # cópia: não altera os registros originais

        # 1. Remover espaços extras nas colunas de texto
        for chave in ["cliente", "produto", "categoria", "regiao"]:
            linha[chave] = linha[chave].strip()

        # 2. Conversão de "data_venda" e descartar datas inválidas
        try:
            linha['data_venda'] = datetime.strptime(linha['data_venda'], "%Y-%m-%d")
        except ValueError:
            relatorio['removidos_data'] += 1
            continue

        # 3. Descartar nulos em "quantidade" e "preco_unitario"
        if linha['quantidade'] == "" or linha['preco_unitario'] == "":
            relatorio['removidos_nulos'] += 1
            continue

        # 4. Ajustar os tipos numéricos
        linha['quantidade'] = int(float(linha['quantidade']))
        linha['preco_unitario'] = float(linha['preco_unitario'])

        # 5. Padronizar "nome do cliente" para o formato Cliente_NNN
        #    - sinaliza quem veio fora do padrão (ex.: "cliente#016", "CLIENTE-016", "Cliente_008!!")
        #    - remove caracteres não alfanuméricos com re.sub()
        #    - reconstrói o nome no formato Cliente_NNN a partir dos 3 dígitos finais
        linha["cliente_fora_do_padrao"] = padrao_cliente.match(linha['cliente']) is None

        nome_limpo = re.sub(r"[^A-Za-z0-9]", "", linha['cliente'])
        numero = re.search(r"\d{3}$", nome_limpo)
        if numero:
            linha['cliente'] = f"Cliente_{numero.group()}"
        else:
            linha['cliente'] = nome_limpo

        if linha["cliente_fora_do_padrao"]:
            relatorio['clientes_padronizados'] += 1

        registros_limpos.append(linha)

    # 6. Montar e imprimir relatório de limpeza
    relatorio["finais"] = len(registros_limpos)
    print("=== RELATÓRIO DE LIMPEZA ===")
    print()
    print(relatorio)
    return registros_limpos, relatorio

In [ ]:
registros_limpos, relatorio = limpar_dados(registros)

### CRIANDO COLUNAS DERIVADAS (RF04)

In [ ]:
def criar_colunas_derivadas(registros):
    """
    Cria as colunas derivadas a partir dos registros limpos:
    receita_total, mes, mes_nome, ano, trimestre e faixa_receita_item.
    Parâmetros:
        registros (list): lista de dicionários já limpos pelo RF03
                          (data_venda como datetime, quantidade int, preco_unitario float).
    Retorna:
        list: os mesmos registros com as novas colunas adicionadas.
    """
    meses_pt = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
        5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
        9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }

    for linha in registros:
        data = linha["data_venda"]

        # Receita total do registro
        receita = round(linha["quantidade"] * linha["preco_unitario"], 2)
        linha["receita_total"] = receita

        # Atributos de data
        linha["mes"] = data.month
        linha["mes_nome"] = meses_pt[data.month]
        linha["ano"] = data.year

        # Trimestre
        if data.month <= 3:
            linha["trimestre"] = "Q1"
        elif data.month <= 6:
            linha["trimestre"] = "Q2"
        elif data.month <= 9:
            linha["trimestre"] = "Q3"
        else:
            linha["trimestre"] = "Q4"

        # Faixa de receita do registro
        if receita < 500:
            linha["faixa_receita_item"] = "Baixo Valor"
        elif receita < 5000:
            linha["faixa_receita_item"] = "Médio Valor"
        else:
            linha["faixa_receita_item"] = "Alto Valor"

    return registros

registros_limpos = criar_colunas_derivadas(registros_limpos)

### CALCULANDO METRICAS AGREGADAS (RF05)

In [ ]:
def calcular_metricas(registros):
    """
    Calcula e exibe as métricas agregadas por mês, trimestre, produto (Top 5), categoria e região.
    Parâmetros:
        registros (list): lista de dicionários com as colunas derivadas do RF04
                          (mes, trimestre, receita_total, quantidade, produto, categoria, regiao).
    Retorna:
        dict: {nome_da_metrica: lista_de_linhas}, com as chaves
              por_mes, por_trimestre, top_produtos, por_categoria e por_regiao.
    """
    metricas_mes = defaultdict(lambda: {"receita_total": 0.0, "quantidade": 0, "n_vendas": 0})
    metricas_trimestre = defaultdict(lambda: {"receita_total": 0.0, "quantidade": 0, "n_vendas": 0})
    metricas_produto = defaultdict(float)
    metricas_categoria = defaultdict(float)
    metricas_regiao = defaultdict(lambda: {"receita_total": 0.0, "n_vendas": 0})

    for r in registros:
        m_num = r["mes"]
        rec = r["receita_total"]
        qtd = r["quantidade"]

        # Agregação por Mês
        metricas_mes[m_num]["receita_total"] += rec
        metricas_mes[m_num]["quantidade"] += qtd
        metricas_mes[m_num]["n_vendas"] += 1

        # Agregação por Trimestre
        metricas_trimestre[r["trimestre"]]["receita_total"] += rec
        metricas_trimestre[r["trimestre"]]["quantidade"] += qtd
        metricas_trimestre[r["trimestre"]]["n_vendas"] += 1

        # Produto e Categoria
        metricas_produto[r["produto"]] += rec
        metricas_categoria[r["categoria"]] += rec

        # Região
        metricas_regiao[r["regiao"]]["receita_total"] += rec
        metricas_regiao[r["regiao"]]["n_vendas"] += 1

    # Estruturação das listas de resultados
    por_mes = [
        {
            "mes": m,
            "receita_total": round(dados["receita_total"], 2),
            "quantidade": dados["quantidade"],
            "n_vendas": dados["n_vendas"]
        }
        for m, dados in sorted(metricas_mes.items())
    ]

    por_trimestre = [
        {
            "trimestre": t,
            "receita_total": round(dados["receita_total"], 2),
            "quantidade": dados["quantidade"],
            "n_vendas": dados["n_vendas"]
        }
        for t, dados in sorted(metricas_trimestre.items())
    ]

    top_produtos = [
        {"produto": p, "receita_total": round(rec, 2)}
        for p, rec in sorted(metricas_produto.items(), key=lambda item: item[1], reverse=True)[:5]
    ]

    por_categoria = [
        {"categoria": c, "receita_total": round(rec, 2)}
        for c, rec in sorted(metricas_categoria.items(), key=lambda item: item[1], reverse=True)
    ]

    por_regiao = [
        {
            "regiao": reg,
            "receita_total": round(dados["receita_total"], 2),
            "ticket_medio": round(dados["receita_total"] / dados["n_vendas"], 2)
        }
        for reg, dados in sorted(metricas_regiao.items(), key=lambda item: item[1]["receita_total"], reverse=True)
    ]

    # === IMPRESSÃO FORMATADA NO CONSOLE / JUPYTER ===

    print("=== POR MES ===")
    print(f"{'mes':>4}  {'receita_total':>13}  {'quantidade':>10}  {'n_vendas':>8}")
    for item in por_mes:
        print(f"{item['mes']:>4}  {item['receita_total']:>13.2f}  {item['quantidade']:>10}  {item['n_vendas']:>8}")
    print()

    print("=== POR TRIMESTRE ===")
    print(f"{'trimestre':>9}  {'receita_total':>13}  {'quantidade':>10}  {'n_vendas':>8}")
    for item in por_trimestre:
        print(f"{item['trimestre']:>9}  {item['receita_total']:>13.2f}  {item['quantidade']:>10}  {item['n_vendas']:>8}")
    print()

    print("=== TOP PRODUTOS ===")
    print(f"{'produto':>20}  {'receita_total':>13}")
    for item in top_produtos:
        print(f"{item['produto']:>20}  {item['receita_total']:>13.2f}")
    print()

    print("=== POR CATEGORIA ===")
    print(f"{'categoria':>15}  {'receita_total':>13}")
    for item in por_categoria:
        print(f"{item['categoria']:>15}  {item['receita_total']:>13.2f}")
    print()

    print("=== POR REGIAO ===")
    print(f"{'regiao':>15}  {'receita_total':>13}  {'ticket_medio':>12}")
    for item in por_regiao:
        print(f"{item['regiao']:>15}  {item['receita_total']:>13.2f}  {item['ticket_medio']:>12.2f}")
    print()

    return {
        "por_mes": por_mes,
        "por_trimestre": por_trimestre,
        "top_produtos": top_produtos,
        "por_categoria": por_categoria,
        "por_regiao": por_regiao
    }

metricas = calcular_metricas(registros_limpos)

### SEGMENTANDO CLIENTES POR NÍVEL DE GASTO (RF06)

In [ ]:
def segmentar_clientes(dataset):
    """
    Agrupa por cliente, soma a receita e classifica em Bronze / Prata / Ouro usando função lambda.
    Faixas: abaixo de R$ 5.000,00 = Bronze | de R$ 5.000,00 a R$ 15.000,00 = Prata | acima de R$ 15.000,00 = Ouro
    Parâmetros:
        dataset (list): lista de dicionários com os registros limpos (com receita_total).
    Retorna:
        list: lista de dicionários com cliente, total_gasto e segmento.
    """
    classificar_cliente = lambda total: "Ouro" if total > 15000 else ("Prata" if total >= 5000 else "Bronze")

    total_por_cliente = {}
    for linha in dataset:
        total_por_cliente[linha['cliente']] = total_por_cliente.get(linha['cliente'], 0) + linha['receita_total']

    clientes = [
        {'cliente': nome, 'total_gasto': round(total, 2), 'segmento': classificar_cliente(total)}
        for nome, total in total_por_cliente.items()
    ]
    return clientes

In [ ]:
def exibir_segmentacao(clientes):
    """
    Exibe os 10 maiores clientes por gasto total e a distribuição de clientes por segmento.
    Parâmetros:
        clientes (list): lista de dicionários com cliente, total_gasto e segmento.
    Retorna:
        list: clientes ordenados por total_gasto em ordem decrescente.
    """
    clientes_ordenados = sorted(clientes, key=lambda x: x['total_gasto'], reverse=True)

    print("=== TOP 10 CLIENTES POR GASTO TOTAL ===")
    print()
    print(f"{'Cliente':<15} {'Total Gasto':>15} {'Segmento':>10}")
    for cliente in clientes_ordenados[:10]:
        print(f"{cliente['cliente']:<15} {cliente['total_gasto']:>15.2f} {cliente['segmento']:>10}")
    print()

    distribuicao_segmentos = defaultdict(int)
    for cliente in clientes:
        distribuicao_segmentos[cliente['segmento']] += 1

    print("=== DISTRIBUIÇÃO DE CLIENTES POR SEGMENTO ===")
    print()
    print(f"{'Segmento':<10} {'Número de Clientes':>20}")
    for segmento, quantidade in distribuicao_segmentos.items():
        print(f"{segmento:<10} {quantidade:>20}")

    return clientes_ordenados

clientes_segmentados = segmentar_clientes(registros_limpos)
clientes_ordenados = exibir_segmentacao(clientes_segmentados)

### ESTATÍSTICAS GERAIS (vendas acima da média)

In [ ]:
def calcular_estatisticas_gerais(registros):
    """
    Calcula estatísticas gerais das vendas, incluindo quantas vendas
    individuais tiveram receita acima da média geral de receita por venda.
    Parâmetros:
        registros (list): lista de dicionários com os registros limpos (com receita_total).
    Retorna:
        dict: estatísticas gerais (apenas valores numéricos).
    """
    total_vendas = len(registros)
    receita_total = sum(linha['receita_total'] for linha in registros)
    receita_media = receita_total / total_vendas if total_vendas else 0
    vendas_acima_media = len([linha for linha in registros if linha['receita_total'] > receita_media])

    estatisticas = {
        'total_vendas': total_vendas,
        'receita_total': round(receita_total, 2),
        'receita_media_por_venda': round(receita_media, 2),
        'vendas_acima_da_media': vendas_acima_media,
    }

    print("=== ESTATÍSTICAS GERAIS ===")
    print()
    print(f"Total de vendas válidas:        {estatisticas['total_vendas']}")
    print(f"Receita total:                  R$ {estatisticas['receita_total']:,.2f}")
    print(f"Receita média por venda:        R$ {estatisticas['receita_media_por_venda']:,.2f}")
    print(f"Vendas acima da média:          {estatisticas['vendas_acima_da_media']}")
    return estatisticas

estatisticas = calcular_estatisticas_gerais(registros_limpos)

In [ ]:
### FUNÇÃO DE ORDEM SUPERIOR (RF07)

In [ ]:
def processar_coluna(registros, coluna, funcao_transformacao, nome_saida=None):
    """
    Aplica uma função de transformação a um campo de cada registro
    (função de ordem superior: recebe outra função como argumento).
    Parâmetros:
        registros (list): lista de dicionários com os registros.
        coluna (str): nome do campo que será transformado.
        funcao_transformacao (function): função aplicada ao valor do campo.
        nome_saida (str, opcional): nome da nova coluna. Padrão: "<coluna>_transformado".
    Retorna:
        list: os registros com a nova coluna adicionada.
    """
    nome_saida = nome_saida or f"{coluna}_transformado"
    for linha in registros:
        linha[nome_saida] = funcao_transformacao(linha[coluna])
    return registros

# Uso 1: converte a receita para milhares de reais
registros_limpos = processar_coluna(registros_limpos, "receita_total",
                                    lambda x: round(x / 1000, 2),
                                    nome_saida="receita_em_milhares")

# Uso 2: classifica o volume da venda pela quantidade
registros_limpos = processar_coluna(registros_limpos, "quantidade",
                                    lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
                                    nome_saida="perfil_volume")

print("=== EXEMPLO DE REGISTRO APÓS processar_coluna ===")
print()
exemplo = registros_limpos[0]
print(f"receita_total: {exemplo['receita_total']}  ->  receita_em_milhares: {exemplo['receita_em_milhares']}")
print(f"quantidade: {exemplo['quantidade']}  ->  perfil_volume: {exemplo['perfil_volume']}")

### EXPORTANDO RESULTADOS EM CSV E JSON (RF08)

In [ ]:
def exportar_resultados(metricas, clientes, estatisticas, pasta_saida="outputs"):
    """
    Exporta os resultados do projeto em CSV e JSON e relê o JSON gravado para conferência.
    Parâmetros:
        metricas (dict): métricas agregadas (precisa da chave "por_mes").
        clientes (list): lista de dicionários com a segmentação de clientes.
        estatisticas (dict): estatísticas gerais (valores numéricos).
        pasta_saida (str): pasta onde os arquivos serão gravados.
    Retorna:
        dict: conteúdo do JSON lido de volta do disco.
    """
    os.makedirs(pasta_saida, exist_ok=True)

    # 1. Métricas por mês em CSV
    caminho_mes = os.path.join(pasta_saida, "metricas_por_mes.csv")
    with open(caminho_mes, "w", newline="", encoding="utf-8-sig") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=metricas["por_mes"][0].keys())
        escritor.writeheader()
        escritor.writerows(metricas["por_mes"])

    # 2. Segmentação de clientes em CSV
    caminho_clientes = os.path.join(pasta_saida, "segmentacao_clientes.csv")
    with open(caminho_clientes, "w", newline="", encoding="utf-8-sig") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=clientes[0].keys())
        escritor.writeheader()
        escritor.writerows(clientes)

    # 3. Estatísticas gerais em JSON
    caminho_json = os.path.join(pasta_saida, "estatisticas_gerais.json")
    with open(caminho_json, "w", encoding="utf-8") as arquivo:
        json.dump(estatisticas, arquivo, indent=4, ensure_ascii=False)

    # 4. Leitura de volta do JSON para confirmar a gravação
    with open(caminho_json, "r", encoding="utf-8") as arquivo:
        conferencia = json.load(arquivo)

    print("=== EXPORTAÇÃO DE RESULTADOS ===")
    print()
    print(f"Gravado: {caminho_mes}")
    print(f"Gravado: {caminho_clientes}")
    print(f"Gravado: {caminho_json}")
    print()
    print(f"JSON gravado e lido de volta: {conferencia}")
    return conferencia

conferencia = exportar_resultados(metricas, clientes_ordenados, estatisticas)

### EXECUTANDO O FLUXO COMPLETO (RF09)

A função `main()` executa todas as etapas acima em sequência, de ponta a ponta: a limpeza acontece antes das transformações e das agregações.

In [ ]:
def main():
    """Executa o fluxo completo do SalesInsight PY, do dataset bruto até a exportação."""
    print("=" * 60)
    print(" SALESINSIGHT PY - Análise de Dados de Vendas")
    print("=" * 60)

    # Etapa 0 - garantir a existência do dataset
    if not os.path.exists("vendas.csv"):
        gerar_dataset_vendas("vendas.csv")

    # Etapa 1 - carregar e inspecionar (RF01, RF02)
    registros = carregar_dataset("vendas.csv")
    inspecionar_dados(registros)
    print()

    # Etapa 2 - limpar (RF03), sempre antes das transformações
    registros_limpos, relatorio = limpar_dados(registros)
    print()

    # Etapa 3 - colunas derivadas (RF04) e função de ordem superior (RF07)
    registros_limpos = criar_colunas_derivadas(registros_limpos)
    registros_limpos = processar_coluna(registros_limpos, "receita_total",
                                        lambda x: round(x / 1000, 2),
                                        nome_saida="receita_em_milhares")
    registros_limpos = processar_coluna(registros_limpos, "quantidade",
                                        lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
                                        nome_saida="perfil_volume")

    # Etapa 4 - métricas agregadas (RF05)
    metricas = calcular_metricas(registros_limpos)

    # Etapa 5 - segmentação de clientes (RF06)
    clientes = segmentar_clientes(registros_limpos)
    clientes_ordenados = exibir_segmentacao(clientes)
    print()

    # Etapa 6 - estatísticas gerais e exportação (RF08)
    estatisticas = calcular_estatisticas_gerais(registros_limpos)
    print()
    exportar_resultados(metricas, clientes_ordenados, estatisticas)

    print("\n[CONCLUÍDO] Fluxo finalizado com sucesso.")

if __name__ == "__main__":
    main()